# کوردی سۆرانی — Kurdish subtitles on a free cloud GPU

**This notebook runs the whole app on Google's free GPU, so your laptop only needs a browser.**
Nothing is installed on your computer and nothing on it gets hot.

### How to use it — three clicks

1. **Runtime → Change runtime type → T4 GPU → Save**
   *(If you skip this, everything still works, but on a slow CPU.)*
2. Press **▶ Run all** (or run the cells top to bottom).
3. Wait until a green box prints **“OPEN THIS LINK”** — click that link.
   The normal Kurdish Sub page opens in a new tab, running on the GPU.

**One permission window will appear** (cell 3): it asks whether the notebook may
save your results to Google Drive. **Allow** is recommended — but **Cancel**
changes nothing about the app; you simply download the files instead.

Keep this Colab tab open while you work. When you are done, close it: the free
machine is deleted automatically.

> Free Colab gives about **12 hours** per session and disconnects after ~90
> minutes of doing nothing. For a normal film you need 10–40 minutes.

## 1 · Check the GPU

You want to see a **Tesla T4**. If it says “command not found”, the GPU is not
switched on: Runtime → Change runtime type → **T4 GPU** → Save, then run this
cell again.

In [ ]:
!nvidia-smi

## 2 · Install Kurdish Sub

Two to four minutes the first time. Everything is downloaded into this temporary
machine; the speech model is downloaded again next time (it is ~1.5 GB, once).

In [ ]:
import os, pathlib, subprocess, sys

REPO = "https://github.com/sahandking86-lgtm/kurdish-sub.git"
BRANCHES = ["main", "arena/190df544-kurdish-sub"]   # main first, this branch as backup
DEST = pathlib.Path("/content/kurdish-sub")

if not (DEST / "app").is_dir():
    for branch in BRANCHES:
        print(f"downloading [{branch}] …")
        r = subprocess.run(["git", "clone", "--depth", "1", "-b", branch, REPO, str(DEST)],
                           capture_output=True, text=True)
        if r.returncode == 0:
            break
        print("  ", (r.stderr or "").strip().splitlines()[-1:] or "")
    else:
        raise SystemExit("Could not download the app — check the internet connection.")

os.chdir(DEST)
print("installing the Python packages (a few minutes) …")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("packages:", "ok" if r.returncode == 0 else "FAILED")

# make sure the GPU can really run the speech model (fixes the classic Colab cuDNN trap)
!python cloud/doctor.py --fix

## 3 · Connect Google Drive — recommended (20 seconds)

A finished film is big. Connecting Drive means a copy of every video and `.srt`
is saved there automatically, so nothing is lost when the free machine switches
off — and you can watch it on your phone.

A Google window will ask for permission: choose your account → **Allow**.
If you would rather not, just run the cell and close the pop-up; everything still
works and you download the result from the app.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Google Drive not connected — that is fine:", exc)

## 4 · Start it and get your link

This prints a **private link**. Open it in a new tab (or on your phone) and use
the app exactly like the desktop version — drop a video, pick the accuracy,
wait, download the result.

*Included:* `--drive` copies every finished video and `.srt` into
`My Drive → KurdishSub` **if** you have connected Drive in this session.
If you have not, results are still downloadable from the app, and from the
folder icon on the left of this notebook (`kurdish-sub-cloud/output/…`).

In [ ]:
import os, pathlib
if not pathlib.Path('cloud/start_cloud.py').exists():
    os.chdir('/content/kurdish-sub')

%run cloud/start_cloud.py --drive


## 5 · Optional — better Kurdish (1 minute)

Kurdish text from **Google Gemini** is clearly better than the keyless Google
Translate engine: more natural sentences, keeps slang and profanity.

Get a free key at <https://aistudio.google.com/apikey>, then paste it in the
app's **gear icon → Translation engine → Google Gemini**. That is all — no need
to touch this notebook.

> Gemini's free tier is not available in every country. If the key is refused,
> stay on *Automatic* (Google Translate works everywhere).

## 6 · Keep the session awake (only for long films)

Colab disconnects a session after about 90 minutes without activity. A normal
film finishes long before that. If you are doing a **very long video** (a 3-hour
film, “Perfect” accuracy), run this cell and leave it running — it prints a dot
every five minutes so Google knows you are still there.

Press the **■ stop** button of this cell when you are finished. The app keeps
working normally while this cell runs.

In [ ]:
import datetime, time
print("Keeping the session awake — press the stop button when you are done.")
try:
    while True:
        time.sleep(300)
        print("still awake", datetime.datetime.now().strftime("%H:%M"), flush=True)
except KeyboardInterrupt:
    print("stopped")

## When you are finished

* **Download** the finished video from the app, or from the file browser on the left.
* Close the browser tab. Google deletes the machine and the models after a while.
* Next time: open this notebook again and press **Run all** — same steps, fresh GPU.

### If something looks wrong

| What you see | What to do |
|---|---|
| `command not found: nvidia-smi` | Runtime → Change runtime type → **T4 GPU** → Save |
| “No GPU available” in Colab | Google is busy. Wait a few minutes, or use a Kaggle account (30 free GPU hours/week) |
| The link does not open | Use the interface shown inside this notebook instead (it is the same app) |
| The notebook disconnected | Run the cells again — that is normal on the free tier |
| Everything is very slow | Check the first cell says **Tesla T4**; pick the “Medium” accuracy level |